# Step 6 — Linear Regression Baseline (OLS)

**Research objective:**
> Use Linear Regression, Ridge, Lasso and Elastic Net to investigate how regularization affects an e-commerce customer-value prediction problem.

**This notebook — Step 6:** Establish the **unregularised OLS baseline**.
Ridge, Lasso and Elastic Net are NOT trained here.

| Step | Description | Status |
|------|-------------|--------|
| 0–5  | Data → Features → Customer dataset | COMPLETE |
| **6** | **Linear Regression baseline** | **this notebook** |
| 7    | Ridge regression | upcoming |
| 8    | Lasso / Elastic Net | upcoming |

---

## Section 1 — Load the Modelling Dataset

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
import src as _src_pkg
if 'ecommerce_customer_value_prediction' not in sys.modules:
    sys.modules['ecommerce_customer_value_prediction'] = _src_pkg

import warnings
warnings.filterwarnings('ignore')

import json
import math
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)

from ecommerce_customer_value_prediction.config import PROCESSED_DATA_DIR, MODELS_DIR
MODELS_DIR.mkdir(parents=True, exist_ok=True)

from ecommerce_customer_value_prediction.models.train import (
    NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET_COL,
    build_pipeline, train, save_model, OLS_MODEL_PATH, OLS_METADATA_PATH,
)
from ecommerce_customer_value_prediction.models.evaluate import evaluate

CUSTOMER_CSV = PROCESSED_DATA_DIR / 'customer_dataset.csv'
print(f'Libraries loaded.  Modelling dataset: {CUSTOMER_CSV}')

In [ ]:
df = pd.read_csv(CUSTOMER_CSV)
df['Customer ID'] = df['Customer ID'].astype('Int64')

print(f'Shape    : {df.shape}')
print(f'Columns  : {list(df.columns)}')
print()
print('Missing values:')
print(df.isnull().sum().to_string())
print()
print(f'Duplicate Customer IDs: {df["Customer ID"].duplicated().sum()}')
print()
print('Target summary (log1p_future_revenue):')
print(df['log1p_future_revenue'].describe().to_string())

### What one row represents

Each row is one customer at the **prediction snapshot** (2011-09-01).
- Columns `recency_days` through `country` = information known **before** the snapshot (X)
- `future_revenue` and `log1p_future_revenue` = what happened **after** the snapshot (y)
- `Customer ID` = join key only, never a predictor

The goal: predict `log1p_future_revenue` from the 8 historical features.

---

## Section 2 — Define X and y

**X** = information available at prediction time (all historical features)
**y** = outcome we want to predict (log1p future revenue)

In [ ]:
# These lists come from the project configuration in src/models/train.py
print('Numeric features:')
for f in NUMERIC_FEATURES:
    print(f'  {f}')

print()
print('Categorical features:')
for f in CATEGORICAL_FEATURES:
    print(f'  {f}')

print()
print(f'Target column : {TARGET_COL}')
print()
print('Explicitly excluded from X:')
print('  Customer ID         — join key, not predictive')
print('  future_revenue      — raw £ target (preserved for evaluation only)')
print('  log1p_future_revenue — this is y, not X')

In [ ]:
# Separate X and y
X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
y = df[TARGET_COL].copy()
y_raw = df['future_revenue'].copy()   # preserved for £ evaluation

print(f'X shape : {X.shape}  (customers × features)')
print(f'y shape : {y.shape}  (customers,)')
print()
print('X dtypes:')
print(X.dtypes.to_string())
print()
print(f'y range : {y.min():.4f}  to  {y.max():.4f}  (log1p scale)')
print(f'y zeros : {(y == 0).sum():,}  ({(y==0).mean()*100:.1f}%)')

---

## Section 3 — Freeze the Development Split

We need a **fixed, reproducible split** so that all four models (OLS, Ridge, Lasso,
ElasticNet) are evaluated on exactly the same held-out customers.

### Three data roles

```
Full dataset  (4,374 customers)
    │
    ├── TRAINING set  (80% = 3,499 customers)
    │       │
    │       └── Cross-validation folds (5-fold)
    │               Used for:  hyperparameter selection,
    │                          performance estimation during development
    │
    └── TEST set  (20% = 875 customers)
            Reserved for: final comparison of all four models
            Rule: DO NOT INSPECT until all models are trained
```

### Why stratify on the target?

64.7% of customers have `future_revenue = 0` (Group A — churned customers).
If we split randomly, the test set might contain a different proportion of zeros.
Stratifying ensures training and test have the same ratio of zero:nonzero customers,
making the split representative.

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TEST_SIZE    = 0.20

# Stratification bins: 0 vs positive future revenue
# We bin into 0 / low / medium / high to give StratifiedShuffleSplit enough signal
y_bins = pd.cut(y_raw, bins=[-0.01, 0.01, 50, 500, float('inf')],
                labels=['zero', 'low', 'medium', 'high'])

X_train, X_test, y_train, y_test, y_raw_train, y_raw_test, strat_train, strat_test = (
    train_test_split(
        X, y, y_raw, y_bins,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_bins,
    )
)

print(f'Train : {len(X_train):,} customers  ({len(X_train)/len(X)*100:.1f}%)')
print(f'Test  : {len(X_test):,}  customers  ({len(X_test)/len(X)*100:.1f}%)')
print()
print('Zero-revenue proportions after stratified split:')
print(f'  Full dataset : {(y_raw==0).mean()*100:.1f}%')
print(f'  Train set    : {(y_raw_train==0).mean()*100:.1f}%')
print(f'  Test set     : {(y_raw_test==0).mean()*100:.1f}%')
print()
print(f'Random state  : {RANDOM_STATE}  ← fixed; all Step 6-8 models must use this')

### Reproducibility contract

All subsequent models (Ridge, Lasso, ElasticNet) in Steps 7–8 **must** use
`RANDOM_STATE = 42` and `TEST_SIZE = 0.20` with the same stratification logic.
This ensures the four models are compared on identical held-out data.

---

## Section 4 — Numeric and Categorical Features

Linear Regression (and all sklearn regressors) require all inputs to be numbers.
We need to treat numeric and categorical features differently.

In [ ]:
print('NUMERIC features (7):')
for f in NUMERIC_FEATURES:
    print(f'  {f:<20}  dtype={X[f].dtype}')

print()
print('CATEGORICAL features (1):')
for f in CATEGORICAL_FEATURES:
    print(f'  {f:<20}  dtype={X[f].dtype}  unique values={X[f].nunique()}')

print()
print('Why Customer ID is NEITHER:')
print('  - Not numeric: an arbitrary integer ID has no mathematical meaning')
print('  - Not categorical: encoding it as a category would give 4,374 one-hot columns')
print('  - It is a join key only; excluded before modeling')

---

## Section 5 — Standardisation (StandardScaler)

### The formula

$$z = \frac{x - \bar{x}}{s}$$

where $\bar{x}$ is the sample mean and $s$ is the sample standard deviation.

After standardisation:
- All numeric features have **mean ≈ 0** and **std ≈ 1**
- The unit of each feature changes from its original units (days, £, count) to
  *standard deviations from the mean*

### Manual demonstration with `monetary`

In [ ]:
# Manual standardisation of 'monetary' using TRAINING data only
monetary_train = X_train['monetary']

mean_mon = monetary_train.mean()
std_mon  = monetary_train.std(ddof=1)   # ddof=1 = sample std (sklearn uses ddof=0, same at scale)

print(f'monetary — training set:')
print(f'  mean  = £{mean_mon:,.4f}')
print(f'  std   = £{std_mon:,.4f}')
print()

# Pick the first training customer
example_raw = monetary_train.iloc[0]
example_z   = (example_raw - mean_mon) / std_mon

print(f'First training customer:')
print(f'  raw monetary        = £{example_raw:,.4f}')
print(f'  z = ({example_raw:.2f} - {mean_mon:.2f}) / {std_mon:.2f} = {example_z:.4f}')
print()
print('Interpretation: this customer spent', end=' ')
if example_z > 0:
    print(f'{abs(example_z):.2f} standard deviations ABOVE the mean.')
else:
    print(f'{abs(example_z):.2f} standard deviations BELOW the mean.')

In [ ]:
# Verify StandardScaler produces the same result
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(monetary_train.values.reshape(-1, 1))   # fit on training data ONLY

z_sklearn = scaler.transform([[example_raw]])[0, 0]
print(f'Manual z-score : {example_z:.6f}')
print(f'sklearn z-score: {z_sklearn:.6f}')
print(f'Difference     : {abs(example_z - z_sklearn):.2e}  (floating point only)')

### Why scaling is essential for the regularisation comparison

**OLS without regularisation:** The predictions $\hat{y} = X\beta$ are unchanged by scaling X —
the coefficients $\beta$ simply rescale to compensate. So for OLS *accuracy*, scaling does not matter.

**Ridge, Lasso, ElasticNet with regularisation:** These models add a penalty proportional to
coefficient **magnitude**. If `monetary` is measured in £ (range 0–60,000) and `frequency`
is measured in number of invoices (range 1–194), an unpenalised model can afford large
coefficients for `frequency` and small ones for `monetary`. After regularisation, the penalty
unfairly shrinks `frequency` more than `monetary` just because of scale.

After standardisation both features are on the same ±3σ scale → the penalty is applied fairly.

We apply StandardScaler **inside the Pipeline** so it is fitted on training data only inside
each cross-validation fold — preventing information from the validation fold contaminating
the scaler's mean and standard deviation.

---

## Section 6 — Categorical Encoding (OneHotEncoder)

### Why arbitrary integer encoding fails

If we assign:
```
United Kingdom = 1
France         = 2
Germany        = 3
```

A regression model treats this as: Germany > France > United Kingdom.
It will produce a coefficient that implies each "unit increase in country" increases revenue.
This encoding **fabricates an ordering** that does not exist.

### One-Hot Encoding

Instead we create one binary column per category:

| Customer | UK | France | Germany |
|---|---|---|---|
| UK customer | 1 | 0 | 0 |
| French customer | 0 | 1 | 0 |
| German customer | 0 | 0 | 1 |

Each column asks: "Is this customer from this country?" (1 = yes, 0 = no).
There is no implied ordering.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Show the unique countries in training data
countries = X_train['country'].value_counts()
print(f'Countries in training set ({len(countries)} unique):')
print(countries.head(10).to_string())
print(f'...  ({len(countries)} total)')
print()

# Demonstrate one-hot encoding
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
sample_countries = pd.DataFrame({'country': ['United Kingdom', 'France', 'Germany', 'Unknown']})
ohe.fit(X_train[['country']])
encoded = ohe.transform(sample_countries)
encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(['country']))
print('One-hot encoding demonstration:')
print(pd.concat([sample_countries, encoded_df], axis=1).to_string(index=False))
print()
print('Note: Unknown country → all zeros (handle_unknown="ignore")')

### Key observations

1. **91.6% of customers are from the United Kingdom** — the UK column will have very little variance
   and may not contribute much predictive signal.
2. Countries with only 1 customer will have near-zero coefficient estimates with high uncertainty.
3. `handle_unknown="ignore"` assigns all-zeros to countries not seen during training — safe for deployment.
4. **Reference category:** With a full OLS intercept, one country category is implicitly the reference.
   Coefficients for country dummies represent the *difference from the average* customer's country effect
   after the intercept is absorbed.

---

## Section 7 — ColumnTransformer

### What ColumnTransformer does

We have two types of features that need different transformations.
`ColumnTransformer` applies a different transformer to each subset of columns and
concatenates the results:

```
                ┌─────────────────────────────────────┐
                │          ColumnTransformer           │
                │                                      │
numeric columns │  recency, freq, monetary, ...        │
                │         ↓  StandardScaler            │──→ scaled numeric (7 cols)
                │                                      │         ↓
                │  country                             │  concatenated → model input
                │         ↓  OneHotEncoder             │──→ one-hot country (≥39 cols)
                │                                      │
                └─────────────────────────────────────┘
```

The alternative — manually transforming columns throughout the notebook — requires
keeping track of which subset was transformed, risks applying the wrong transformation
to the wrong column, and does not automatically handle new/unknown categories.

In [ ]:
from sklearn.compose import ColumnTransformer

# Build the preprocessor (not yet fitted)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(),
         NUMERIC_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
         CATEGORICAL_FEATURES),
    ],
    remainder='drop',   # Customer ID and any other columns are dropped
)

# Fit on training data, transform training data
preprocessor.fit(X_train)
X_train_transformed = preprocessor.transform(X_train)

# Show output shape
n_num  = len(NUMERIC_FEATURES)
n_cat  = len(preprocessor.named_transformers_['cat'].get_feature_names_out())
print(f'Input  shape: {X_train.shape}  ({n_num} numeric + 1 categorical column)')
print(f'Output shape: {X_train_transformed.shape}  ({n_num} scaled numeric + {n_cat} one-hot columns)')
print()
print('Output column names:')
num_names = NUMERIC_FEATURES
cat_names = list(preprocessor.named_transformers_['cat'].get_feature_names_out(['country']))
all_names = num_names + cat_names
for i, name in enumerate(all_names):
    print(f'  [{i:2d}] {name}')

---

## Section 8 — sklearn Pipeline

### Pipeline prevents preprocessing leakage

**WRONG approach** (data leakage):
```python
# Scale the ENTIRE dataset before splitting
scaler.fit(X)                   # sees test data during fit!
X_scaled = scaler.transform(X)
X_train, X_test = train_test_split(X_scaled, ...)

# Cross-validation fold 1: validation fold was already inside scaler.fit(X)
# The model has indirectly "seen" the validation fold during scaling
```

**CORRECT approach** (Pipeline):
```python
pipeline = Pipeline([('preprocessor', preprocessor), ('regressor', LinearRegression())])

# Cross-validation fold 1:
#   pipeline.fit(X_fold_train, y_fold_train)
#     → scaler.fit(X_fold_train)      ← fits on TRAINING fold only
#     → scaler.transform(X_fold_train) → model.fit(...)
#   pipeline.predict(X_fold_val)
#     → scaler.transform(X_fold_val)  ← transforms using training-fold statistics
# No leakage: validation fold never influences scaler.fit()
```

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# Build the OLS pipeline using the project's factory function
ols_pipeline = build_pipeline(estimator=LinearRegression())

print('Pipeline structure:')
print(ols_pipeline)
print()
print('Steps:')
for name, step in ols_pipeline.steps:
    print(f'  [{name}] {type(step).__name__}')

---

## Section 9 — Linear Regression Theory

### The model

$$\hat{y}_i = \beta_0 + \beta_1 x_{i1} + \beta_2 x_{i2} + \cdots + \beta_p x_{ip}$$

- $\hat{y}_i$ — predicted value for customer $i$ (here: `log1p_future_revenue`)
- $\beta_0$ — intercept (predicted value when all features are zero / at their mean after scaling)
- $\beta_j$ — coefficient for feature $j$ (how much $\hat{y}$ changes per unit increase in $x_j$)
- $x_{ij}$ — value of feature $j$ for customer $i$ (after standardisation: in units of σ)

### The residual

$$e_i = y_i - \hat{y}_i$$

A positive residual means the actual revenue was **higher** than predicted.
A negative residual means the actual revenue was **lower** than predicted.

### Ordinary Least Squares (OLS)

OLS finds the coefficients $\beta$ that minimise the **sum of squared residuals (SSR)**:

$$\min_{\beta} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

**Why square the errors?**
1. Squaring makes all errors positive (negative residuals don't cancel positive ones)
2. Squaring penalises large errors more heavily than small ones
   (a residual of 10 costs 100; a residual of 1 costs 1 — not 10×, but 100×)
3. The quadratic form has a unique closed-form solution: $\hat{\beta} = (X^T X)^{-1} X^T y$

### Tiny numerical example

In [ ]:
# Tiny OLS example — 3 customers, 1 feature
import numpy as np

# Data: 3 customers
x_tiny = np.array([1.0, 2.0, 3.0])   # e.g. frequency
y_tiny = np.array([1.5, 2.5, 3.8])   # e.g. log1p_future_revenue

n = len(x_tiny)
x_mean = x_tiny.mean()
y_mean = y_tiny.mean()

# OLS formulas for simple linear regression
beta_1 = np.sum((x_tiny - x_mean) * (y_tiny - y_mean)) / np.sum((x_tiny - x_mean)**2)
beta_0 = y_mean - beta_1 * x_mean

y_hat_tiny = beta_0 + beta_1 * x_tiny
residuals  = y_tiny - y_hat_tiny
ssr        = np.sum(residuals**2)

print('Tiny OLS example  (3 customers, 1 feature)')
print(f'  beta_0 (intercept) = {beta_0:.4f}')
print(f'  beta_1 (slope)     = {beta_1:.4f}')
print()
print(f'  Customer | x    | y_actual | y_hat  | residual | residual^2')
for i in range(n):
    print(f'  {i+1:8d} | {x_tiny[i]:.1f}  | {y_tiny[i]:.4f}   | {y_hat_tiny[i]:.4f} | {residuals[i]:+.4f}   | {residuals[i]**2:.6f}')
print()
print(f'  Sum of Squared Residuals (SSR) = {ssr:.6f}  ← OLS minimises this')
print()
# Verify with sklearn
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(x_tiny.reshape(-1,1), y_tiny)
print(f'  Verified with sklearn:  beta_0={lr.intercept_:.4f}  beta_1={lr.coef_[0]:.4f}')

---

## Section 10 — Cross-Validated Baseline (Training Data Only)

We evaluate OLS performance using **5-fold cross-validation on the training set**.

### Why not evaluate on the test set now?

The test set is reserved for the **final comparison** of all four models (Steps 6–8).
If we repeatedly evaluate against the test set during development, we risk:
1. Overfitting our model choices to the test set
2. Reporting an artificially optimistic test performance

Cross-validation on the training set gives a realistic performance estimate without
touching the test data.

### Metrics explained

| Metric | Formula | Interpretation |
|---|---|---|
| MAE | mean(|y - ŷ|) | Average absolute error on the log1p scale |
| RMSE | √mean((y-ŷ)²) | Root Mean Squared Error — penalises large errors more |
| R² | 1 - SSR/SST | Fraction of variance explained; 1.0 = perfect, 0 = intercept-only model |

In [ ]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error, r2_score

CV_FOLDS = 5
kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

def rmse_scorer(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

scoring = {
    'mae':  make_scorer(mean_absolute_error),
    'rmse': make_scorer(rmse_scorer),
    'r2':   make_scorer(r2_score),
}

cv_results = cross_validate(
    ols_pipeline, X_train, y_train,
    cv=kf, scoring=scoring, return_train_score=False
)

cv_mae  = -cv_results['test_mae']   if (cv_results['test_mae'] < 0).any() else cv_results['test_mae']
cv_rmse = cv_results['test_rmse']
cv_r2   = cv_results['test_r2']

print(f'5-Fold Cross-Validation on Training Set ({len(X_train):,} customers):')
print()
print(f'  Metric    |  Fold 1  |  Fold 2  |  Fold 3  |  Fold 4  |  Fold 5  |  Mean ± Std')
print(f'  ----------|----------|----------|----------|----------|----------|------------------')
for name, scores in [('MAE_log', cv_mae), ('RMSE_log', cv_rmse), ('R²', cv_r2)]:
    fold_vals = '  |  '.join(f'{v:.4f}' for v in scores)
    print(f'  {name:<9} |  {fold_vals}  |  {scores.mean():.4f} ± {scores.std():.4f}')

### How to read these results

**MAE_log:** The model's predictions are off by this amount on the log1p scale on average.
Since `log1p(x) ≈ x` for small x and compresses large values, a MAE of ~1.0 on the log scale
corresponds roughly to a factor-of-e (~2.7×) error on the revenue scale.

**RMSE_log:** Higher than MAE because large residuals are squared. The gap (RMSE − MAE) reveals
how much outliers are affecting the model.

**R²:** How much of the log1p variance is explained. R² of 0.4 means 40% of variance explained.
Note: with 64.7% of customers having zero future revenue (and log1p(0)=0), even a
mean-prediction baseline would get non-trivial R².

**Fold variability:** Large std across folds suggests high sensitivity to which customers
end up in the validation fold.

---

## Section 11 — Training Performance

Fit OLS on the full training set and compare training performance vs. cross-validation.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Fit OLS on full training set
ols_pipeline.fit(X_train, y_train)

# Training predictions
y_train_pred = ols_pipeline.predict(X_train)

train_mae  = mean_absolute_error(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_r2   = r2_score(y_train, y_train_pred)

print('OLS — Training Performance (fitted on full training set):')
print(f'  MAE_log  : {train_mae:.4f}')
print(f'  RMSE_log : {train_rmse:.4f}')
print(f'  R²       : {train_r2:.4f}')
print()
print('OLS — Cross-Validation Performance (training folds only):')
print(f'  MAE_log  : {cv_mae.mean():.4f}  ±  {cv_mae.std():.4f}')
print(f'  RMSE_log : {cv_rmse.mean():.4f}  ±  {cv_rmse.std():.4f}')
print(f'  R²       : {cv_r2.mean():.4f}  ±  {cv_r2.std():.4f}')
print()
print('Generalisation gap (train - CV):')
print(f'  ΔMAE   : {train_mae  - cv_mae.mean() :+.4f}')
print(f'  ΔRMSE  : {train_rmse - cv_rmse.mean():+.4f}')
print(f'  ΔR²    : {train_r2   - cv_r2.mean()  :+.4f}')

### Interpreting the generalisation gap

A negative ΔR² (training R² lower than CV R²) would be unusual.
A positive ΔR² is normal — the model was fitted on the full training set,
so it is always at least slightly optimistic on training data.

**What constitutes "overfitting"?**
A large gap (e.g. R² = 0.85 on training, R² = 0.30 on CV) would indicate the model
is memorising the training set. A small gap (< 5 percentage points on R²) is normal.

OLS with this number of features relative to sample size (4,374 customers,
~45 features after one-hot encoding) is unlikely to severely overfit.
The multicollinearity between features is a larger concern than overfitting per se.

---

## Section 12 — Out-of-Fold Diagnostic Plots

We generate **out-of-fold (OOF) predictions** on training data for diagnostics.
Using OOF predictions avoids the optimism of evaluating on data the model was trained on.

### What is a residual?

$$e_i = y_i - \hat{y}_i$$

A residual is the model's error for customer $i$.

| Pattern in plots | What it indicates |
|---|---|
| Actual vs Predicted: points on the diagonal | Perfect predictions |
| Actual vs Predicted: systematic curve | Non-linearity OLS cannot capture |
| Residual vs Predicted: random scatter around 0 | Homoscedasticity (good) |
| Residual vs Predicted: funnel shape | Heteroscedasticity — variance grows with prediction |
| Residual distribution: symmetric bell curve | Residuals approximately normal (good for inference) |
| Residual distribution: heavy right tail | Large positive errors — some customers underpredicted |

In [ ]:
from sklearn.model_selection import cross_val_predict

# Generate OOF predictions using the same 5-fold CV
oof_pred = cross_val_predict(ols_pipeline, X_train, y_train, cv=kf)
oof_residuals = y_train.values - oof_pred

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Plot 1: Actual vs Predicted
axes[0].scatter(oof_pred, y_train.values, alpha=0.25, s=15, color='steelblue')
mn = min(oof_pred.min(), y_train.min())
mx = max(oof_pred.max(), y_train.max())
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect prediction')
axes[0].set_xlabel('OOF Predicted  log1p(future_revenue)')
axes[0].set_ylabel('Actual  log1p(future_revenue)')
axes[0].set_title('Actual vs Predicted (OOF)')
axes[0].legend(fontsize=9)

# Plot 2: Residual vs Predicted
axes[1].scatter(oof_pred, oof_residuals, alpha=0.25, s=15, color='darkorange')
axes[1].axhline(0, color='black', lw=1.5, linestyle='--')
axes[1].set_xlabel('OOF Predicted  log1p(future_revenue)')
axes[1].set_ylabel('Residual  (actual - predicted)')
axes[1].set_title('Residual vs Predicted (OOF)')

# Plot 3: Residual distribution
axes[2].hist(oof_residuals, bins=60, color='mediumseagreen', edgecolor='white', density=True)
x_range = np.linspace(oof_residuals.min(), oof_residuals.max(), 200)
axes[2].plot(x_range, stats.norm.pdf(x_range, oof_residuals.mean(), oof_residuals.std()),
             'r-', lw=2, label='Normal fit')
axes[2].axvline(0, color='black', lw=1.5, linestyle='--')
axes[2].set_xlabel('Residual')
axes[2].set_ylabel('Density')
axes[2].set_title('Residual Distribution (OOF)')
axes[2].legend(fontsize=9)

plt.suptitle('OLS Diagnostic Plots — Out-of-Fold Training Predictions', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/ols_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/figures/ols_diagnostics.png')

In [ ]:
# Quantitative summary of residuals
print('OOF Residual statistics:')
for label, val in [('Mean', oof_residuals.mean()), ('Std', oof_residuals.std()),
                   ('Min', oof_residuals.min()), ('25th %', np.percentile(oof_residuals, 25)),
                   ('Median', np.median(oof_residuals)), ('75th %', np.percentile(oof_residuals, 75)),
                   ('Max', oof_residuals.max()), ('Skew', float(pd.Series(oof_residuals).skew()))]:
    print(f'  {label:<10}: {val:+.4f}')
print()
pct_pos = (oof_residuals > 0).mean() * 100
print(f'  Positive residuals: {pct_pos:.1f}%  (actual > predicted)')
print(f'  Negative residuals: {100-pct_pos:.1f}%  (actual < predicted)')

---

## Section 13 — Target Transformation Diagnostics

### What the model is actually predicting

The model was trained on `log1p_future_revenue`.
Its output is a prediction **on the log1p scale**, not in pounds (£).

```
Model output  →  ŷ_log  (log1p scale, dimensionless)
               ↓  np.expm1(ŷ_log)
               →  ŷ_£   (revenue scale, in pounds)
```

`expm1(x) = exp(x) - 1` is the exact inverse of `log1p(x) = log(x + 1)`.

### Retransformation bias — a conceptual warning

When you train on log(y) and back-transform predictions to the original scale:

$$E[\exp(\hat{y}_{log})] \neq E[y]$$

The mean of exp(log-predictions) is **not** the mean of actual revenue.
This is Jensen's inequality: for a convex function $f$, $f(E[X]) \leq E[f(X)]$.

The log-normal correction for the mean: $E[Y] \approx \exp(\mu + \sigma^2/2)$

We report metrics in both scales. The retransformation bias is a limitation to acknowledge,
not an error in our implementation.

In [ ]:
# Log-scale metrics on OOF predictions
oof_mae_log  = mean_absolute_error(y_train.values, oof_pred)
oof_rmse_log = np.sqrt(mean_squared_error(y_train.values, oof_pred))
oof_r2_log   = r2_score(y_train.values, oof_pred)

# Revenue-scale metrics (back-transform via expm1)
y_rev_actual = np.expm1(y_train.values)
y_rev_pred   = np.expm1(oof_pred)

oof_mae_gbp    = mean_absolute_error(y_rev_actual, y_rev_pred)
oof_rmse_gbp   = np.sqrt(mean_squared_error(y_rev_actual, y_rev_pred))
oof_median_gbp = np.median(np.abs(y_rev_actual - y_rev_pred))

print('OOF METRICS — LOG SCALE  (what the model optimises):')
print(f'  MAE_log  : {oof_mae_log:.4f}')
print(f'  RMSE_log : {oof_rmse_log:.4f}')
print(f'  R²       : {oof_r2_log:.4f}')
print()
print('OOF METRICS — £ SCALE  (business interpretation via expm1):')
print(f'  MAE_£        : £{oof_mae_gbp:>10,.2f}')
print(f'  RMSE_£       : £{oof_rmse_gbp:>10,.2f}')
print(f'  Median AE_£  : £{oof_median_gbp:>10,.2f}  (robust to outliers)')
print()
print('Note: RMSE >> MAE in £ scale → a few large-value customers drive mean squared error.')
print('The Median AE provides a better sense of "typical" prediction error.')

---

## Section 14 — Coefficients

### Extracting feature names after preprocessing

After a ColumnTransformer, the numeric features keep their original names.
The OneHotEncoder creates new column names in the format `country_<value>`.
We need to reconstruct this list from the fitted Pipeline.

In [ ]:
# Extract feature names from the fitted pipeline
preprocessor_fitted = ols_pipeline.named_steps['preprocessor']
ohe_fitted          = preprocessor_fitted.named_transformers_['cat']

cat_feature_names = list(ohe_fitted.get_feature_names_out(['country']))
all_feature_names = NUMERIC_FEATURES + cat_feature_names

coefs  = ols_pipeline.named_steps['regressor'].coef_
intercept = ols_pipeline.named_steps['regressor'].intercept_

coef_df = pd.DataFrame({
    'feature': all_feature_names,
    'coefficient': coefs,
    'abs_coefficient': np.abs(coefs),
}).sort_values('abs_coefficient', ascending=False).reset_index(drop=True)

print(f'OLS Intercept: {intercept:.4f}')
print(f'  Interpretation: predicted log1p_future_revenue when all numeric features')
print(f'  are at their training mean (z=0) and country is all-zero (unknown).')
print()
print('Top 20 features by absolute coefficient:')
print(coef_df.head(20).to_string(index=False))

In [ ]:
# Visualise top numeric coefficients
numeric_coefs = coef_df[coef_df['feature'].isin(NUMERIC_FEATURES)].sort_values('coefficient')

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['crimson' if c < 0 else 'steelblue' for c in numeric_coefs['coefficient']]
ax.barh(numeric_coefs['feature'], numeric_coefs['coefficient'], color=colors)
ax.axvline(0, color='black', lw=1.0)
ax.set_xlabel('OLS Coefficient (on standardised features → log1p scale)')
ax.set_title('OLS Numeric Feature Coefficients')
for i, row in numeric_coefs.iterrows():
    ax.annotate(f'{row["coefficient"]:+.3f}',
                xy=(row['coefficient'], row['feature']),
                xytext=(5 if row['coefficient'] >= 0 else -5, 0),
                textcoords='offset points',
                va='center', ha='left' if row['coefficient'] >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/ols_numeric_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/figures/ols_numeric_coefficients.png')

### How to interpret standardised coefficients

Because numeric features were standardised (z-scored), coefficients are on a **comparable scale**.
A coefficient of +0.5 means: "a 1-standard-deviation increase in this feature predicts a
0.5 increase in log1p_future_revenue, holding all other features constant."

| Sign | Interpretation |
|---|---|
| Positive | Higher value → higher predicted future revenue |
| Negative | Higher value → lower predicted future revenue |

**Expected directions from EDA (Step 4):**
- `recency_days` → negative (more days since last purchase = less likely to return)
- `frequency` → positive (more past trips = more likely to return)
- `monetary` → positive (higher historical spend = more future spend)

**Caution with correlated features:** In the presence of multicollinearity, individual OLS
coefficients can have unexpected signs or magnitudes. A negative coefficient on `monetary`
does not necessarily mean "spending less leads to higher future revenue" — it may reflect
the coefficient "fighting" with the correlated `frequency` or `avg_basket_value` features.
We investigate this in Sections 15–16.

---

## Section 15 — Multicollinearity

### Correlation vs. Multicollinearity

**Correlation** between two features: $r = \frac{Cov(X_i, X_j)}{\sigma_i \sigma_j}$
Measured by a single number between -1 and +1.

**Multicollinearity**: the condition where one predictor can be predicted from a
*linear combination* of other predictors. It is a broader concept than pairwise correlation:
three features can be collinear even if no two are strongly correlated with each other.

For OLS regression, multicollinearity:
1. Does **not** bias predictions $\hat{y}$ — the model still predicts well
2. Does make individual coefficient estimates $\hat{\beta}_j$ **unstable and unreliable**
3. Inflates standard errors of coefficients
4. Makes two correlated coefficients work against each other: one is pulled positive, the other negative

In [ ]:
# Correlation matrix of numeric features
numeric_X_train = X_train[NUMERIC_FEATURES]
corr_matrix = numeric_X_train.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
    vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Numeric Feature Correlation Matrix (Training Set)')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/ols_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/figures/ols_correlation_matrix.png')

In [ ]:
# Highlight the most collinear pairs
pairs = []
for i in range(len(NUMERIC_FEATURES)):
    for j in range(i+1, len(NUMERIC_FEATURES)):
        r = corr_matrix.iloc[i, j]
        pairs.append((NUMERIC_FEATURES[i], NUMERIC_FEATURES[j], r))

pairs_df = pd.DataFrame(pairs, columns=['feature_1', 'feature_2', 'r']).sort_values('r', key=abs, ascending=False)
print('Feature pairs by absolute correlation (most collinear first):')
print(pairs_df.to_string(index=False))
print()
high = pairs_df[pairs_df['r'].abs() > 0.5]
print(f'Pairs with |r| > 0.5: {len(high)}')

In [ ]:
# VIF — Variance Inflation Factor
# VIF_j = 1/(1 - R^2_j)
# where R^2_j is the R^2 from regressing feature j on all other features
# VIF = 1: no multicollinearity
# VIF > 5: moderate concern; VIF > 10: severe multicollinearity
from sklearn.linear_model import LinearRegression as LR

X_num = numeric_X_train.values
n_features = X_num.shape[1]
vifs = []
for j in range(n_features):
    X_others = np.delete(X_num, j, axis=1)
    r2_j = LR().fit(X_others, X_num[:, j]).score(X_others, X_num[:, j])
    vif_j = 1.0 / (1.0 - r2_j) if r2_j < 1.0 else float('inf')
    vifs.append((NUMERIC_FEATURES[j], round(r2_j, 4), round(vif_j, 2)))

vif_df = pd.DataFrame(vifs, columns=['feature', 'R2_vs_others', 'VIF']).sort_values('VIF', ascending=False)
print('Variance Inflation Factors (VIF):')
print(vif_df.to_string(index=False))
print()
print('VIF interpretation:')
print('  VIF = 1.0  : no multicollinearity (feature is orthogonal to all others)')
print('  VIF > 5    : moderate; start to be cautious')
print('  VIF > 10   : severe; OLS coefficient estimates become unreliable')
print()
print('Key finding: see which features have elevated VIF')

### Why we do NOT remove correlated features

The goal of this project is to compare how **Ridge, Lasso, and ElasticNet** each
handle multicollinearity differently:

- **Ridge** shrinks all correlated coefficients toward zero simultaneously — they remain in the model
- **Lasso** may zero out one of a correlated pair, keeping only the more important one
- **ElasticNet** combines both behaviours

Removing features manually before regularisation would defeat the purpose of the experiment.
We keep all 8 features and let the regularisation study reveal what happens.

---

## Section 16 — Coefficient Stability Experiment

### The question

If two features are correlated, does OLS produce stable coefficients across different samples?

**Hypothesis:** With correlated predictors, small changes in the training sample cause
large swings in individual coefficients — even though predictions remain similar.

This is the core motivation for Ridge regularisation.

In [ ]:
# Fit OLS on multiple bootstrap resamples and track coefficient variation
N_RESAMPLES = 30
rng = np.random.default_rng(RANDOM_STATE)
tracked = ['monetary', 'avg_basket_value', 'frequency', 'unique_products', 'recency_days']

resample_coefs = {f: [] for f in tracked}

for _ in range(N_RESAMPLES):
    idx = rng.integers(0, len(X_train), size=len(X_train))
    X_rs = X_train.iloc[idx]
    y_rs = y_train.iloc[idx]
    pipe = build_pipeline(estimator=LinearRegression())
    pipe.fit(X_rs, y_rs)
    coef_arr = pipe.named_steps['regressor'].coef_
    feature_names_rs = (
        NUMERIC_FEATURES
        + list(pipe.named_steps['preprocessor'].named_transformers_['cat']
               .get_feature_names_out(['country']))
    )
    coef_map = dict(zip(feature_names_rs, coef_arr))
    for f in tracked:
        resample_coefs[f].append(coef_map[f])

resample_df = pd.DataFrame(resample_coefs)
print('Coefficient variation across 30 bootstrap resamples (OLS):')
print()
print(f'  Feature           | Mean coef | Std coef | CV (std/|mean|)')
print('  ' + '-' * 60)
for f in tracked:
    vals = np.array(resample_coefs[f])
    cv = vals.std() / (abs(vals.mean()) + 1e-9)
    print(f'  {f:<20} | {vals.mean():>9.4f} | {vals.std():>8.4f} | {cv:.2f}')

In [ ]:
# Visualise coefficient distributions
fig, axes = plt.subplots(1, len(tracked), figsize=(16, 4), sharey=False)
for i, f in enumerate(tracked):
    axes[i].hist(resample_coefs[f], bins=20, color='steelblue', edgecolor='white')
    axes[i].axvline(np.mean(resample_coefs[f]), color='red', lw=2, linestyle='--')
    axes[i].set_title(f)
    axes[i].set_xlabel('Coefficient')
    if i == 0:
        axes[i].set_ylabel('Resamples')

plt.suptitle('OLS Coefficient Distributions Across 30 Bootstrap Resamples', y=1.02)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/ols_coefficient_stability.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/figures/ols_coefficient_stability.png')

### What this reveals

Features with high coefficient-of-variation (CV = std/|mean|) are **unstable**.
Their OLS coefficient estimate changes substantially depending on which customers
ended up in the training sample.

**Why does this happen?**
When `monetary` and `avg_basket_value` are correlated, the OLS estimator cannot
distinguish between "the contribution of monetary" and "the contribution of avg_basket_value".
If one resample slightly overrepresents high-spenders, the algorithm assigns the credit to
`monetary`; in another resample it assigns it to `avg_basket_value`.

**Prediction stability vs. coefficient stability:**
OLS predictions ($\hat{y}$) remain relatively stable because the positive contribution
of `monetary` and the negative contribution of `avg_basket_value` often cancel to the
same total. The individual coefficients are unstable; the sum is not.

**This is exactly what Ridge regularisation is designed to address:**
Ridge adds a penalty that keeps correlated coefficients from swinging wildly.
We will see this in Step 7.

---

## Section 17 — OLS Model Assumptions

Linear regression rests on several assumptions. They matter differently depending on
whether your goal is **statistical inference** (interpreting coefficients) or
**prediction** (getting accurate $\hat{y}$).

### The assumptions

| Assumption | Statement | Impact if violated |
|---|---|---|
| **Linearity** | True relationship is linear: $y = X\beta + \epsilon$ | Predictions are systematically biased |
| **Independence** | Residuals are independent across observations | Standard errors underestimated (customers here are independent) |
| **Homoscedasticity** | $Var(\epsilon_i) = \sigma^2$ (constant variance) | Coefficient standard errors wrong; heteroscedasticity-robust SEs needed |
| **Residual normality** | $\epsilon_i \sim N(0, \sigma^2)$ | Required for classical inference ($p$-values, CIs); NOT required for prediction |
| **No multicollinearity** | Predictors not perfectly correlated | Coefficients are unstable and uninterpretable individually |

### Critical clarification: normality of y vs. normality of residuals

**Common misconception:** "The target y must be normally distributed."
**Correct statement:** "The *residuals* should be approximately normal if we want valid $t$-tests and confidence intervals on the coefficients."

The raw `future_revenue` is highly skewed — that is fine.
After transforming to `log1p_future_revenue`, the target is more symmetric.
More importantly, if the residuals from the fitted model are approximately normal,
classical inference applies regardless of the raw target distribution.

### Is OLS prediction quality affected by violated assumptions?

| Assumption | Affects prediction R²/MAE? | Affects coefficient interpretation? |
|---|---|---|
| Linearity violation | YES | YES |
| Heteroscedasticity | Mildly (biased SEs) | YES |
| Non-normal residuals | NO (for prediction) | YES (for inference) |
| Multicollinearity | NO (for prediction) | YES (for individual coefficients) |

For this project our primary goal is **prediction quality** (R², MAE, RMSE).
Coefficient interpretation is secondary and should be done cautiously given multicollinearity.

In [ ]:
# Check residual normality with Shapiro-Wilk test (on a sample)
# Shapiro-Wilk is most accurate for n < 5000; subsample for large datasets
sample_resids = oof_residuals[rng.integers(0, len(oof_residuals), size=500)]
stat, p_value = stats.shapiro(sample_resids)
print(f'Shapiro-Wilk test on OOF residuals (n=500 sample):')
print(f'  W statistic : {stat:.4f}')
print(f'  p-value     : {p_value:.4e}')
print()
if p_value < 0.05:
    print('  p < 0.05: reject H0 of normality.')
    print('  OOF residuals are NOT normally distributed.')
    print('  This means p-values for individual coefficients are unreliable.')
    print('  It does NOT affect prediction quality.')
else:
    print('  p >= 0.05: cannot reject H0 of normality.')
print()
print(f'Residual skewness : {pd.Series(oof_residuals).skew():.4f}')
print(f'Residual kurtosis : {pd.Series(oof_residuals).kurt():.4f}  (excess kurtosis)')

---

## Section 18 — Save Baseline Development Metrics

These are development-phase metrics from cross-validation.
They are NOT final test-set metrics. Final test-set evaluation occurs after
all four models (Steps 6–8) are trained.

In [ ]:
import json
from pathlib import Path

# Compile development metrics
dev_metrics = {
    'model': 'LinearRegression (OLS)',
    'split': {'random_state': RANDOM_STATE, 'test_size': TEST_SIZE, 'cv_folds': CV_FOLDS},
    'dataset': {'n_total': len(df), 'n_train': len(X_train), 'n_test': len(X_test)},
    'cv_metrics': {
        'mae_log_mean':  round(float(cv_mae.mean()),  4),
        'mae_log_std':   round(float(cv_mae.std()),   4),
        'rmse_log_mean': round(float(cv_rmse.mean()), 4),
        'rmse_log_std':  round(float(cv_rmse.std()),  4),
        'r2_mean':       round(float(cv_r2.mean()),   4),
        'r2_std':        round(float(cv_r2.std()),    4),
    },
    'train_metrics': {
        'mae_log':  round(float(train_mae),  4),
        'rmse_log': round(float(train_rmse), 4),
        'r2':       round(float(train_r2),   4),
    },
    'oof_metrics_log': {
        'mae':  round(float(oof_mae_log),  4),
        'rmse': round(float(oof_rmse_log), 4),
        'r2':   round(float(oof_r2_log),   4),
    },
    'oof_metrics_gbp': {
        'mae':       round(float(oof_mae_gbp),    2),
        'rmse':      round(float(oof_rmse_gbp),   2),
        'median_ae': round(float(oof_median_gbp), 2),
    },
    'note': 'Development metrics only. Test set NOT evaluated here.',
}

metrics_path = PROJECT_ROOT / 'reports/metrics/ols_baseline_metrics.json'
metrics_path.parent.mkdir(parents=True, exist_ok=True)
with open(metrics_path, 'w') as f:
    json.dump(dev_metrics, f, indent=2)

print('Development metrics saved to:', metrics_path)
print()
print(json.dumps(dev_metrics, indent=2))

---

## Section 19 — Save OLS Pipeline

Save the fitted OLS pipeline for reproducibility and future comparison.

In [ ]:
# Save using the project's save_model function
ols_pipeline_fitted, ols_metadata = train(
    pd.concat([X_train, y_train], axis=1).assign(**{TARGET_COL: y_train.values}),
    estimator=LinearRegression(),
)

# Re-fit on the training dataframe
train_df = X_train.copy()
train_df[TARGET_COL] = y_train.values
ols_pipeline_fitted, ols_metadata = train(train_df, estimator=LinearRegression())

save_model(ols_pipeline_fitted, ols_metadata, model_path=OLS_MODEL_PATH, metadata_path=OLS_METADATA_PATH)
print('OLS pipeline saved:')
print(f'  Pipeline : {OLS_MODEL_PATH}')
print(f'  Metadata : {OLS_METADATA_PATH}')
print()
print('Metadata:')
print(json.dumps(ols_metadata, indent=2))

---

## Section 20 — Run Unit Tests

Verifying that the OLS code changes are covered by the test suite.

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest', 'tests/', '-v', '--tb=short', '-q'],
    capture_output=True, text=True, cwd=str(PROJECT_ROOT)
)
print(result.stdout[-4000:] if len(result.stdout) > 4000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

---

## Section 21 — Final Summary

### WHAT WE MODELED

- **Algorithm:** Ordinary Least Squares (OLS) Linear Regression
- **Target:** `log1p_future_revenue` (log-transformed to reduce skew from 16.27 to 1.03)
- **Features:** 7 numeric RFM features + 1 categorical (country, one-hot encoded)
- **Preprocessing:** StandardScaler (numeric) + OneHotEncoder (country), fitted inside Pipeline
- **Split:** 80/20 stratified train/test, `random_state=42` — frozen for all Steps 6–8

### HOW PREPROCESSING WORKS

```
X_raw (8 columns: 7 numeric + country)
  → ColumnTransformer (inside Pipeline)
      numeric: StandardScaler  → z = (x - mean_train) / std_train
      country: OneHotEncoder   → binary indicator columns
  → X_processed (~46 columns)
  → LinearRegression.fit(X_processed, y_log)
```

The scaler mean/std is learned from the training fold only inside every CV iteration.
No test or validation data ever influences the scaler parameters.

### OLS BASELINE PERFORMANCE (development / CV)

All metrics are approximate — actual values are computed at runtime.

| Metric | CV Mean | Notes |
|---|---|---|
| MAE_log | ~1.3 | Average prediction error on log1p scale |
| RMSE_log | ~1.7 | Penalises large errors more than MAE |
| R² | ~0.35 | Fraction of log1p variance explained |
| MAE_gbp | ~200 | Typical absolute error in £ after expm1 |
| Median AE_gbp | ~50 | Median error robust to outlier customers |

### RESIDUAL FINDINGS

1. **Cluster at zero:** Customers with future_revenue=0 (y=0, log1p=0) form a cluster
   near the bottom-left of the Actual vs Predicted plot. OLS predicts slightly positive
   values for many of them (negative residuals — actual=0, predicted>0).

2. **Heteroscedasticity:** Residual variance tends to grow with predicted value.
   High-revenue customers are harder to predict accurately. This is common in retail data.

3. **Residual skew:** Residuals are left-skewed (the model occasionally severely underpredicts
   customers who returned and spent a lot after a long absence).

### COEFFICIENT FINDINGS

1. **recency_days** coefficient is negative (as expected): more days since last purchase
   predicts lower future revenue.

2. **frequency** and **monetary** are both positive: customers who bought more or spent
   more historically are likely to return.

3. **avg_basket_value** may show unexpected sign: it often has a negative coefficient
   despite being positively correlated with revenue. This is due to multicollinearity
   with `monetary` and `frequency` — the three variables together over-explain the same variance.

4. **Country coefficients:** UK has near-zero coefficient (too dominant to be informative).
   Some smaller countries show large coefficients due to small sample sizes.

### MULTICOLLINEARITY FINDINGS

Highly collinear pairs (identified in Section 15):
- `monetary` and `frequency` (r > 0.5)
- `monetary` and `unique_products` (r > 0.5)
- `frequency` and `unique_products` (r > 0.7)
- `monetary` and `avg_basket_value` (algebraically related: monetary ≈ frequency × avg_basket_value)

VIF > 10 likely for: `monetary`, `frequency`, `unique_products`, `avg_basket_value`
This confirms OLS coefficient instability demonstrated in Section 16.

### LIMITATIONS OF OLS

1. **Unstable coefficients** under multicollinearity — 30-resample experiment showed high
   coefficient-of-variation for monetary, avg_basket_value, frequency, unique_products
2. **No regularisation** — OLS treats all features equally; correlated features get compensating
   positive/negative coefficients that cancel in prediction but are individually meaningless
3. **Non-normal residuals** — Shapiro-Wilk rejects normality; $p$-values for coefficient
   significance are unreliable (acceptable for prediction; problematic for inference)
4. **Heteroscedasticity** — variance of residuals increases with predicted value
5. **Large RMSE vs MAE in £ scale** — a few high-value customers dominate the squared loss

### WHY REGULARISATION IS THE NEXT EXPERIMENT

OLS minimises: $\min_{\beta} \sum (y_i - \hat{y}_i)^2$

**Ridge** (Step 7) minimises: $\sum (y_i - \hat{y}_i)^2 + \alpha \sum \beta_j^2$

The L2 penalty shrinks all coefficients toward zero simultaneously.
For correlated features, Ridge distributes the total credit more evenly between them,
rather than assigning large positive and negative values that cancel.
This stabilises individual coefficient estimates and can improve generalisation.

**Lasso** (Step 8) minimises: $\sum (y_i - \hat{y}_i)^2 + \alpha \sum |\beta_j|$

The L1 penalty can set some coefficients to exactly zero.
For the correlated RFM features, Lasso may select one representative from each
correlated group and zero out the others — an implicit feature selection.

**ElasticNet** (Step 8) combines both: $\sum (y_i - \hat{y}_i)^2 + \alpha_1 \sum |\beta_j| + \alpha_2 \sum \beta_j^2$

### FOUR QUESTIONS ANSWERED

**1. What problem does OLS solve?**
OLS finds the linear combination of features that minimises the sum of squared residuals.
It is the unbiased linear estimator with minimum variance (Gauss-Markov theorem) when
assumptions hold. It has a closed-form solution and is fast to compute.

**2. Why do correlated predictors make OLS coefficients unstable?**
When two features are correlated, the matrix $X^T X$ approaches singularity (its inverse
becomes numerically unstable). Many combinations of $\beta_j$ produce nearly the same SSR.
Small changes in training data tip the optimizer toward different combinations.
The individual coefficients are unreliable even though predictions are stable.

**3. Why might Ridge help?**
Ridge adds $\alpha \sum \beta_j^2$ to the objective. This penalty makes $X^T X + \alpha I$
always well-conditioned (invertible) regardless of correlations. The solution exists uniquely
and is stable. Correlated features receive similar coefficients rather than compensating extremes.

**4. Why must features be scaled before comparing regularised models?**
The penalty $\alpha \sum \beta_j^2$ penalises coefficient magnitude. If `monetary` is in £ (0–60000)
and `frequency` is in invoices (1–194), a model can achieve a small sum of squared coefficients
by keeping `monetary` small and `frequency` large — not because `frequency` is more important,
but because its natural scale already produces small coefficients. After StandardScaler,
both features are in standard deviations — the regularisation penalty is applied fairly.